## Google Colab Rust Setup

The following cell is used to set up a Rust environment on Colab. Don't execute it locally!

Many thanks to [`mateusvmv`](https://github.com/mateusvmv) for this hack in [`gist.github.com/korakot/ae95315ea6a3a3b33ee26203998a59a3`](https://gist.github.com/korakot/ae95315ea6a3a3b33ee26203998a59a3?permalink_comment_id=4715636#gistcomment-4715636).

In [ ]:
# This script sets up and spins up a Jupyter Notebook environment with a Rust kernel using Nix and IPC Proxy. 
!wget -qO- https://gist.github.com/wiseaidev/2af6bef753d48565d11bcd478728c979/archive/3f6df40db09f3517ade41997b541b81f0976c12e.tar.gz | tar xvz --strip-components=1
!bash setup_evcxr_kernel.sh

## Install Required Dependencies

In [2]:
:dep plotters = { version = "^0.3.7", default-features = false, features = ["evcxr", "all_series", "all_elements"] }
// or
// :dep plotters = { git = "https://github.com/plotters-rs/plotters" }

In [3]:
:dep ndarray = {version = "0.17.2"}
// or
// :dep ndarray = { git = "https://github.com/rust-ndarray/ndarray" }

In [4]:
:dep ndarray-rand = {version = "0.16.0"}
// or
// :dep ndarray-rand = { git = "https://github.com/rust-ndarray/ndarray/tree/ndarray-rand-0.16.0" }

### Import Modules

In [5]:
use ndarray::Array1;
use ndarray_rand::RandomExt;
use ndarray_rand::rand_distr::StandardNormal;
use plotters::prelude::*;

### Synthetic Data

In [6]:
let n_points = 30_000usize;

let wind_speed: Vec<f64> = Array1::<f64>::random(n_points, StandardNormal)
  .mapv(|v| 12.0 + 2.5 * v)
  .to_vec();

// Temperature: N(32, 8)
let temperature: Vec<f64> = Array1::<f64>::random(n_points, StandardNormal)
  .mapv(|v| 32.0 + 8.0 * v)
  .to_vec();

// Humidity: N(80, 12)
let humidity: Vec<f64> = Array1::<f64>::random(n_points, StandardNormal)
  .mapv(|v| 80.0 + 12.0 * v)
  .to_vec();

## 1. Histograms

### 1.1 Stepfilled histogram

In [7]:
evcxr_figure((640, 480), |root| {
    _ = root.fill(&WHITE);

    let n_bins = 50usize;

    let (lo, hi) = wind_speed
        .iter()
        .fold((f64::MAX, f64::MIN), |(lo, hi), &v| (lo.min(v), hi.max(v)));

    let width = (hi - lo) / n_bins as f64;

    let mut counts = vec![0usize; n_bins];

    for &v in &wind_speed {
        let idx = ((v - lo) / width).floor() as usize;
        counts[idx.min(n_bins - 1)] += 1;
    }

    let total = wind_speed.len() as f64;
    let density: Vec<f64> = counts.iter().map(|&c| c as f64 / (total * width)).collect();
    let max_d = density.iter().cloned().fold(0.0f64, f64::max);

    let mut chart = ChartBuilder::on(&root)
        .caption("Wind Speed Distribution, Stepfilled Histogram", ("Arial", 24).into_font())
        .margin(20)
        .x_label_area_size(50)
        .y_label_area_size(50)
        .build_cartesian_2d(lo..hi, 0.0..max_d * 1.15)?;

    chart.configure_mesh()
        .x_desc("Wind Speed (m/s)")
        .y_desc("Probability Density")
        .axis_desc_style(("sans-serif", 16))
        .draw()?;

    let fill_color = RGBColor(240, 130, 110);
    let edge_color = RGBColor(200, 40, 30);

    let hist1 = density.iter().enumerate().map(|(i, &d)| {
        let x0 = lo + i as f64 * width;
        let x1 = x0 + width;
        Rectangle::new([(x0, 0.0), (x1, d)], fill_color.filled())
    });

    let hist2 = density.iter().enumerate().map(|(i, &d)| {
        let x0 = lo + i as f64 * width;
        let x1 = x0 + width;
        Rectangle::new(
            [(x0, 0.0), (x1, d)],
            ShapeStyle {
                color: edge_color.to_rgba(),
                filled: false,
                stroke_width: 1,
            },
        )
    });

    chart.draw_series(hist1)?;

    chart.draw_series(hist2)?;
    
    Ok(())
}).style("width:100%")

Wind Speed Distribution, Stepfilled Histogram
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
Probability Density
 
 
Wind Speed (m/s)
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
0.0
 
 
 
0.02
 
 
 
0.04
 
 
 
0.06
 
 
 
0.08
 
 
 
0.1
 
 
 
0.12
 
 
 
0.14
 
 
 
0.16
 
 
 
0.18
 
 
 
 
2.0
 
 
 
4.0
 
 
 
6.0
 
 
 
8.0
 
 
 
10.0
 
 
 
12.0
 
 
 
14.0
 
 
 
16.0
 
 
 
18.0
 
 
 
20.0
 
 
 
22.0

### 1.2 Step Outline Histogram

In [8]:
evcxr_figure((640, 480), |root| {
    _ = root.fill(&WHITE);

    let n_bins = 50usize;

    let (lo, hi) = wind_speed
        .iter()
        .fold((f64::MAX, f64::MIN), |(lo, hi), &v| (lo.min(v), hi.max(v)));

    let width = (hi - lo) / n_bins as f64;

    let mut counts = vec![0usize; n_bins];

    for &v in &wind_speed {
        let idx = ((v - lo) / width).floor() as usize;
        counts[idx.min(n_bins - 1)] += 1;
    }

    let total = wind_speed.len() as f64;
    let density: Vec<f64> = counts.iter().map(|&c| c as f64 / (total * width)).collect();
    let max_d = density.iter().cloned().fold(0.0f64, f64::max);

    let mut chart = ChartBuilder::on(&root)
        .caption("Wind Speed Distribution, Step Outline Histogram", ("Arial", 24).into_font())
        .margin(20)
        .x_label_area_size(50)
        .y_label_area_size(50)
        .build_cartesian_2d(lo..hi, 0.0..max_d * 1.15)?;

    chart.configure_mesh()
        .x_desc("Wind Speed (m/s)")
        .y_desc("Probability Density")
        .axis_desc_style(("sans-serif", 16))
        .draw()?;

    let mut pts: Vec<(f64, f64)> = vec![(lo, 0.0)];
    for (i, &d) in density.iter().enumerate() {
        let x0 = lo + i as f64 * width;
        let x1 = x0 + width;
        pts.push((x0, d));
        pts.push((x1, d));
    }
    if let Some(&(last_x, _)) = pts.last() {
        pts.push((last_x, 0.0));
    }

    let indigo = RGBColor(55, 0, 179);

    let hist = std::iter::once(PathElement::new(
        pts,
        ShapeStyle {
            color: indigo.to_rgba(),
            filled: false,
            stroke_width: 2,
        },
    ));

    chart.draw_series(hist)?;
    
    Ok(())
}).style("width:100%")

Wind Speed Distribution, Step Outline Histogram
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
Probability Density
 
 
Wind Speed (m/s)
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
0.0
 
 
 
0.02
 
 
 
0.04
 
 
 
0.06
 
 
 
0.08
 
 
 
0.1
 
 
 
0.12
 
 
 
0.14
 
 
 
0.16
 
 
 
0.18
 
 
 
 
2.0
 
 
 
4.0
 
 
 
6.0
 
 
 
8.0
 
 
 
10.0
 
 
 
12.0
 
 
 
14.0
 
 
 
16.0
 
 
 
18.0
 
 
 
20.0
 
 
 
22.0
 
 
<polyline fill="none" opacity="1" stroke="#3700B3" stroke-width="2" points="70,409 70,409 80,409 80,409 91,409 91,409 102,409 102,409 113,409 113,409 124,409 124,409 135,409 135,409 146,409 146,408 157,408 157,407 168,407 168,405 179,405 179,400 190,400 190,400 201,400 201,393 212,393 212,380 223,380 223,368 234,368 234,359 245,359 245,333 256,333 256,308 267,308 267,279 278,279 278,247 289,247 289,222 300,222 300,191 311,191 311,146 322,146 322,143 333,143 333,120 344,120 344,113 355,113 355,97 366,97 366,112 377,112 377,139 388,139 388,166 399,166 399,187 410,187 410,238 421,238 421,261 432,261 432,285 443,285 443,312 454,312 454,341 465,341 465,356 476,356 476,375 487,375 487,383 498,383 498,392 509,392 509,398 520,398 520,405 531,405 531,406 542,406 542,406 553,406 553,408 564,408 564,408 575,408 575,409 586,409 586,409 597,409 597,409 608,409 608,409 619,409 619,409 "/>

### 1.3 2-D histogram

In [9]:
fn plasma_cmap(t: f64) -> RGBColor {
    let t = t.clamp(0.0, 1.0);
    let r = (255.0 * (0.05 + 0.95 * t)) as u8;
    let g = (255.0 * (0.1 * (1.0 - t) + 0.1 * t * t)) as u8;
    let b = (255.0 * ((1.0 - t) * 0.9 + t * 0.15)) as u8;
    RGBColor(r.min(255), g.min(255), b.min(255))
}

In [10]:
/// 2-D histogram of (temperature, humidity) data.
evcxr_figure((1280, 720), |root| {
    _ = root.fill(&WHITE);
    
    let (main_area, bar_area) = root.split_horizontally(700);

    let n_bins = 50usize;

    let x_min = temperature.iter().cloned().fold(f64::MAX, f64::min);
    let x_max = temperature.iter().cloned().fold(f64::MIN, f64::max);
    let y_min = humidity.iter().cloned().fold(f64::MAX, f64::min);
    let y_max = humidity.iter().cloned().fold(f64::MIN, f64::max);
    let x_w = (x_max - x_min) / n_bins as f64;
    let y_w = (y_max - y_min) / n_bins as f64;

    let mut grid = vec![0u32; n_bins * n_bins];
    for (&xv, &yv) in temperature.iter().zip(humidity.iter()) {
        let ix = ((xv - x_min) / x_w).floor() as usize;
        let iy = ((yv - y_min) / y_w).floor() as usize;
        grid[iy.min(n_bins - 1) * n_bins + ix.min(n_bins - 1)] += 1;
    }
    let max_count = *grid.iter().max().unwrap_or(&1);

    let mut chart = ChartBuilder::on(&main_area)
        .caption("2-D Histogram: Temperature vs Humidity", ("Arial", 24).into_font())
        .margin(20)
        .x_label_area_size(50)
        .y_label_area_size(50)
        .build_cartesian_2d(x_min..x_max, y_min..y_max)?;

    chart.configure_mesh()
        .x_desc("Temperature (°C)")
        .y_desc("Relative Humidity (%)")
        .axis_desc_style(("sans-serif", 20).into_font().color(&BLACK))
        .label_style(("sans-serif", 15).into_font().color(&BLACK))
        .draw()?;


    let hist: Vec<Rectangle<(f64, f64)>> = (0..n_bins)
        .flat_map(|iy| {
            let grid = &grid;
            (0..n_bins).map(move |ix| {
                let t = grid[iy * n_bins + ix] as f64 / max_count as f64;
                let color = plasma_cmap(t);
                let x0 = x_min + ix as f64 * x_w;
                let y0 = y_min + iy as f64 * y_w;
                Rectangle::new([(x0, y0), (x0 + x_w, y0 + y_w)], color.filled())
            })
        })
        .collect();

    chart.draw_series(hist)?;

    let bar = bar_area.margin(80, 20, 30, 10);
    let bar_h = 580u32;
    for i in 0..bar_h {
        let t = 1.0 - i as f64 / bar_h as f64;
        let y = 20 + i as i32;
        bar.draw(&Rectangle::new(
            [(5, y), (32, y + 1)],
            plasma_cmap(t).filled(),
        ))?;
    }
    let style = ("sans-serif", 20)
        .into_font()
        .color(&BLACK)
        .into_text_style(&bar);
    bar.draw_text(&format!("{}", max_count), &style, (35, 20))?;
    bar.draw_text("0", &style, (35, 20 + bar_h as i32 - 10))?;

    Ok(())
}).style("width:100%")

2-D Histogram: Temperature vs Humidity
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
Relative Humidity (%)
 
 
Temperature (°C)
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
40.0
 
 
 
50.0
 
 
 
60.0
 
 
 
70.0
 
 
 
80.0
 
 
 
90.0
 
 
 
100.0
 
 
 
110.0
 
 
 
120.0
 
 
 
 
0.0
 
 
 
10.0
 
 
 
20.0
 
 
 
30.0
 
 
 
40.0
 
 
 
50.0
 
 
 
60.0
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 


## 2. Contour Plots

### 2.1 Simple Labeled Contour

### Helpers 

In [11]:
use std::f64::consts::PI;

/// Build contour grid: Z = sin(X) * cos(Y) over [-π, π] × [-π, π].
fn build_contour_grid(step: f64) -> (Vec<f64>, Vec<f64>, Vec<f64>, usize, usize) {
    let xs: Vec<f64> = {
        let mut v = Vec::new();
        let mut x = -PI;
        while x <= PI {
            v.push(x);
            x += step;
        }
        v
    };
    let ys = xs.clone();
    let cols = xs.len();
    let rows = ys.len();
    let mut z = vec![0.0f64; cols * rows];
    for (j, &yv) in ys.iter().enumerate() {
        for (i, &xv) in xs.iter().enumerate() {
            z[j * cols + i] = xv.sin() * yv.cos();
        }
    }
    (xs, ys, z, cols, rows)
}

// Contour level set for sin(X)*cos(Y): from -0.8 to +0.8
fn contour_levels() -> Vec<f64> {
    (-4..=4)
        .filter(|&k| k != 0)
        .map(|k| k as f64 * 0.2)
        .collect()
}

/// "Inferno" inspired map: black → deep purple → orange → yellow.
fn inferno_cmap(t: f64) -> RGBColor {
    let t = t.clamp(0.0, 1.0);
    let r = (255.0 * (0.5 - (2.0 * t - 1.0).powi(2)).max(0.0) + 255.0 * t.powi(3)) as u8;
    let g = (255.0 * (t * t * 0.85)) as u8;
    let b = (255.0 * ((1.0 - t) * 0.6 * (1.0 - t))) as u8;
    RGBColor(r.min(255), g.min(255), b.min(255))
}

/// Viridis-like map: purple → teal → yellow-green.
fn viridis_cmap(t: f64) -> RGBColor {
    let t = t.clamp(0.0, 1.0);
    let r = (255.0 * (0.267 + 0.733 * t * t)) as u8;
    let g = (255.0 * (0.005 + 0.873 * t * (1.0 - 0.3 * t))) as u8;
    let b = (255.0 * (0.329 * (1.0 - t * 1.1).max(0.0) + 0.05)) as u8;
    RGBColor(r.min(255), g.min(255), b.min(255))
}

fn lerp(a: f64, b: f64, t: f64) -> f64 {
    a + (b - a) * t
}

fn marching_squares_segments(
    z: &[f64],
    cols: usize,
    rows: usize,
    xs: &[f64],
    ys: &[f64],
    level: f64,
) -> Vec<((f64, f64), (f64, f64))> {
    let mut segs = Vec::new();
    for j in 0..rows - 1 {
        for i in 0..cols - 1 {
            let v00 = z[j * cols + i];
            let v10 = z[j * cols + i + 1];
            let v01 = z[(j + 1) * cols + i];
            let v11 = z[(j + 1) * cols + i + 1];
            let mut case = 0u8;
            if v00 >= level {
                case |= 1;
            }
            if v10 >= level {
                case |= 2;
            }
            if v11 >= level {
                case |= 4;
            }
            if v01 >= level {
                case |= 8;
            }
            if case == 0 || case == 15 {
                continue;
            }

            let (x0, x1, y0, y1) = (xs[i], xs[i + 1], ys[j], ys[j + 1]);
            let frac = |a: f64, b: f64| -> f64 {
                if (b - a).abs() < 1e-12 {
                    0.5
                } else {
                    (level - a) / (b - a)
                }
            };
            let bottom = (lerp(x0, x1, frac(v00, v10)), y0);
            let right = (x1, lerp(y0, y1, frac(v10, v11)));
            let top = (lerp(x0, x1, frac(v01, v11)), y1);
            let left = (x0, lerp(y0, y1, frac(v00, v01)));

            match case {
                1 | 14 => segs.push((bottom, left)),
                2 | 13 => segs.push((bottom, right)),
                3 | 12 => segs.push((left, right)),
                4 | 11 => segs.push((right, top)),
                5 => {
                    segs.push((bottom, left));
                    segs.push((right, top));
                }
                6 | 9 => segs.push((bottom, top)),
                7 | 8 => segs.push((left, top)),
                10 => {
                    segs.push((bottom, right));
                    segs.push((left, top));
                }
                _ => {}
            }
        }
    }
    segs
}

fn chain_segments(segs: &[((f64, f64), (f64, f64))]) -> Vec<Vec<(f64, f64)>> {
    if segs.is_empty() {
        return vec![];
    }
    let mut used = vec![false; segs.len()];
    let mut chains: Vec<Vec<(f64, f64)>> = Vec::new();
    let eps = 1e-6;
    let close = |a: (f64, f64), b: (f64, f64)| (a.0 - b.0).abs() < eps && (a.1 - b.1).abs() < eps;

    for start in 0..segs.len() {
        if used[start] {
            continue;
        }
        used[start] = true;
        let mut chain = vec![segs[start].0, segs[start].1];
        loop {
            let tail = *chain.last().unwrap();
            let mut found = false;
            for i in 0..segs.len() {
                if used[i] {
                    continue;
                }
                if close(segs[i].0, tail) {
                    chain.push(segs[i].1);
                    used[i] = true;
                    found = true;
                    break;
                } else if close(segs[i].1, tail) {
                    chain.push(segs[i].0);
                    used[i] = true;
                    found = true;
                    break;
                }
            }
            if !found {
                break;
            }
        }
        chains.push(chain);
    }
    chains
}


### Contour Plot

In [12]:
evcxr_figure((640, 480), |root| {
    _ = root.fill(&WHITE);

    let (xs, ys, z, cols, rows) = build_contour_grid(0.04);
    let levels = contour_levels();

    let mut chart = ChartBuilder::on(&root)
        .caption("Contour: Z = sin(X)·cos(Y)", ("Arial", 24).into_font())
        .margin(20)
        .x_label_area_size(50)
        .y_label_area_size(50)
        .build_cartesian_2d(-PI..PI, -PI..PI)?;

    chart.configure_mesh()
        .x_desc("X (radians)")
        .y_desc("Y (radians)")
        .axis_desc_style(("sans-serif", 20))
        .draw()?;


    let n = levels.len();
    for (li, &level) in levels.iter().enumerate() {
        let t = li as f64 / (n - 1) as f64;
        let color = viridis_cmap(t);
        let segs = marching_squares_segments(&z, cols, rows, &xs, &ys, level);
        let chains = chain_segments(&segs);
        for chain in &chains {
            chart.draw_series(std::iter::once(PathElement::new(
                chain.clone(),
                color.stroke_width(2),
            )))?;
        }
        if let Some(longest) = chains.iter().max_by_key(|c| c.len()) {
            let mid = longest[longest.len() / 2];
            chart.draw_series(std::iter::once(Text::new(
                format!("{:.1}", level),
                (mid.0, mid.1),
                ("sans-serif", 18).into_font().color(&BLACK),
            )))?;
        }
    }
    
    Ok(())
}).style("width:100%")

Contour: Z = sin(X)·cos(Y)
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
Y (radians)
 
 
X (radians)
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
-3.0
 
 
 
-2.0
 
 
 
-1.0
 
 
 
0.0
 
 
 
1.0
 
 
 
2.0
 
 
 
3.0
 
 
 
 
-3.0
 
 
 
-2.0
 
 
 
-1.0
 
 
 
0.0
 
 
 
1.0
 
 
 
2.0
 
 
 
3.0
 
 
<polyline fill="none" opacity="1" stroke="#440160" stroke-width="2" points="425,409 425,407 425,405 426,403 426,402 427,400 427,398 429,396 429,394 430,394 432,391 433,390 434,389 436,387 436,386 439,384 440,384 442,382 443,381 446,380 447,380 450,378 452,378 454,377 457,376 460,375 461,375 464,374 468,374 471,373 475,373 475,373 478,373 482,373 485,373 487,373 489,373 492,373 496,374 499,374 503,375 503,375 506,376 510,377 511,378 513,379 516,380 517,380 520,382 520,382 524,384 524,384 527,387 527,387 529,389 531,391 531,391 533,394 534,396 534,397 535,398 536,400 537,403 537,405 537,407 537,409 "/>
<polyline fill="none" opacity="1" stroke="#440160" stroke-width="2" points="192,266 191,266 188,265 185,264 181,263 181,263 178,262 175,261 174,261 171,259 170,259 167,257 166,256 164,255 163,254 160,252 160,252 158,250 157,248 156,247 155,245 154,243 153,242 153,240 152,238 151,236 151,234 151,231 151,229 151,227 151,224 152,222 152,220 153,218 153,218 155,215 156,213 157,212 158,211 160,208 160,208 163,206 164,205 166,204 167,203 170,202 171,201 174,199 174,199 178,198 181,197 181,197 185,196 188,195 190,195 192,194 195,194 199,194 202,193 206,193 209,193 213,193 216,194 220,194 223,195 223,195 227,195 230,196 233,197 234,197 237,198 239,199 241,200 244,202 244,202 248,204 248,204 251,206 251,206 253,208 255,210 256,211 257,213 258,214 259,215 260,218 261,220 262,222 262,222 262,224 263,227 263,229 263,231 263,234 262,236 262,238 262,238 261,240 260,243 259,245 258,246 257,247 255,250 255,250 253,252 251,254 251,254 248,256 247,256 244,258 244,259 241,260 239,261 237,262 234,263 232,263 230,264 227,265 223,265 222,266 220,266 216,266 213,267 209,267 206,267 202,267 199,266 195,266 192,266 "/>
 
 
 
-0.8
 
<polyline fill="none" opacity="1" stroke="#471F53" stroke-width="2" points="400,409 400,407 400,405 401,403 401,400 402,398 402,398 402,396 403,394 404,391 405,389 405,389 406,387 407,384 409,383 409,382 411,380 412,378 413,378 415,375 416,375 418,373 419,372 421,371 423,370 425,368 426,368 429,366 429,366 433,364 435,364 436,363 440,362 442,362 443,361 447,360 450,359 451,359 454,359 457,358 461,358 464,357 468,357 469,357 471,357 475,357 478,357 482,357 485,357 489,357 492,357 493,357 496,357 499,357 503,358 506,358 510,359 511,359 513,360 517,360 520,361 521,362 524,362 527,364 528,364 531,365 533,366 534,367 538,368 538,368 541,371 541,371 545,373 545,373 547,375 548,376 550,378 552,380 552,380 554,382 555,384 555,385 556,387 558,389 559,391 559,391 560,394 560,396 561,398 561,400 562,403 562,405 562,407 562,409 "/>
<polyline fill="none" opacity="1" stroke="#471F53" stroke-width="2" points="185,282 184,282 181,281 178,280 174,280 172,279 171,279 167,278 164,277 164,277 160,276 158,275 157,274 153,273 153,272 150,271 148,270 146,269 145,268 143,266 142,266 139,263 139,263 137,261 136,259 135,259 134,256 132,254 132,254 131,252 130,250 129,247 129,247 128,245 127,243 127,240 126,238 126,236 126,234 126,231 126,229 126,227 126,224 126,222 127,220 127,218 128,215 129,213 129,213 130,211 131,208 132,206 132,206 134,204 135,202 136,201 137,199 139,197 139,197 142,195 143,194 145,192 146,191 148,190 150,189 152,188 153,187 157,186 157,186 160,184 163,183 164,183 167,182 171,181 171,181 174,180 178,180 181,179 183,179 185,178 188,178 192,178 195,177 199,177 202,177 206,177 209,177 213,177 216,177 220,178 223,178 227,178 230,179 230,179 234,179 237,180 241,181 242,181 244,181 248,182 250,183 251,184 255,185 256,186 258,186 261,188 262,188 265,190 265,

### 2.2 Contour with Viridis Colour Bar

In [13]:
evcxr_figure((1280, 720), |root| {
    _ = root.fill(&WHITE);

    let (xs, ys, z, cols, rows) = build_contour_grid(0.04);
    let levels = contour_levels();

    let (main_area, bar_area) = root.split_horizontally(820);

    let mut chart = ChartBuilder::on(&main_area)
        .caption("Contour with Colour Bar: Z = sin(X)·cos(Y)", ("Arial", 24).into_font())
        .margin(20)
        .x_label_area_size(50)
        .y_label_area_size(50)
        .build_cartesian_2d(-PI..PI, -PI..PI)?;

    chart.configure_mesh()
        .x_desc("X (radians)")
        .y_desc("Y (radians)")
        .axis_desc_style(("sans-serif", 20))
        .draw()?;

    let n = levels.len();
    for (li, &level) in levels.iter().enumerate() {
        let t = li as f64 / (n - 1) as f64;
        let color = viridis_cmap(t);
        let segs = marching_squares_segments(&z, cols, rows, &xs, &ys, level);
        let chains = chain_segments(&segs);
        for chain in &chains {
            chart.draw_series(std::iter::once(PathElement::new(
                chain.clone(),
                color.stroke_width(2),
            )))?;
        }
        if let Some(longest) = chains.iter().max_by_key(|c| c.len()) {
            let mid = longest[longest.len() / 2];
            chart.draw_series(std::iter::once(Text::new(
                format!("{:.1}", level),
                (mid.0, mid.1),
                ("sans-serif", 18).into_font().color(&BLACK),
            )))?;
        }
    }

    let bar = bar_area.margin(70, 15, 35, 10);
    let bar_h = 560u32;
    for i in 0..bar_h {
        let t = 1.0 - i as f64 / bar_h as f64;
        let y = 20 + i as i32;
        bar.draw(&Rectangle::new(
            [(5, y), (30, y + 1)],
            viridis_cmap(t).filled(),
        ))?;
    }
    let style = ("sans-serif", 24).into_text_style(&bar);
    bar.draw_text(&format!("{:.1}", levels.last().unwrap()), &style, (33, 20))?;
    bar.draw_text(
        &format!("{:.1}", levels.first().unwrap()),
        &style,
        (33, 20 + bar_h as i32 - 10),
    )?;

    Ok(())
}).style("width:100%")

Contour with Colour Bar: Z = sin(X)·cos(Y)
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
Y (radians)
 
 
X (radians)
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
-3.0
 
 
 
-2.0
 
 
 
-1.0
 
 
 
0.0
 
 
 
1.0
 
 
 
2.0
 
 
 
3.0
 
 
 
 
-3.0
 
 
 
-2.0
 
 
 
-1.0
 
 
 
0.0
 
 
 
1.0
 
 
 
2.0
 
 
 
3.0
 
 
<polyline fill="none" opacity="1" stroke="#440160" stroke-width="2" points="542,649 542,646 542,642 543,638 543,637 544,634 545,630 546,627 548,624 548,623 550,619 552,616 553,615 556,611 557,611 560,608 561,606 564,604 566,603 570,600 571,600 575,597 577,596 580,595 585,593 588,592 589,592 594,591 599,590 603,589 608,588 608,588 612,588 617,588 622,588 624,588 626,589 631,589 636,590 640,591 645,592 645,592 650,594 654,596 655,596 659,598 662,600 664,601 668,604 668,604 673,608 673,608 676,611 677,613 680,615 682,619 682,619 684,623 686,627 687,628 688,630 689,634 690,638 690,642 691,646 691,649 "/>
<polyline fill="none" opacity="1" stroke="#440160" stroke-width="2" points="232,409 231,409 227,408 223,407 218,406 218,405 213,404 209,402 209,401 204,399 203,398 199,395 198,394 195,391 194,390 190,386 190,386 187,383 186,380 185,379 183,375 181,371 181,371 180,367 179,363 178,360 177,356 177,352 177,348 177,344 178,341 179,337 180,333 181,329 181,329 183,325 185,322 186,320 187,318 190,314 190,314 193,310 195,309 197,306 199,305 202,302 204,301 209,299 209,299 213,296 217,295 218,294 223,293 227,292 230,291 232,291 237,290 241,289 246,289 250,289 255,289 260,289 264,289 269,290 274,291 274,291 278,292 283,294 286,295 288,295 292,297 295,299 297,300 301,302 302,303 306,306 306,306 310,310 311,311 314,314 315,316 317,318 319,322 320,324 321,325 323,329 324,333 325,336 325,337 326,341 326,344 326,348 326,352 326,356 326,360 325,363 325,364 324,367 322,371 321,375 320,376 319,379 316,383 315,384 313,386 311,389 310,390 306,394 306,394 302,397 301,398 297,400 294,402 292,403 288,405 286,405 283,406 278,408 274,409 272,409 269,410 264,411 260,411 255,411 250,411 246,411 241,411 237,410 232,409 "/>
 
 
 
-0.8
 
<polyline fill="none" opacity="1" stroke="#471F53" stroke-width="2" points="509,649 509,646 509,642 509,638 510,634 510,631 510,630 511,627 512,623 513,619 515,615 515,614 516,611 518,608 520,605 520,604 523,600 524,597 525,596 528,592 529,592 532,588 534,587 536,585 538,583 541,581 543,580 547,577 548,577 552,574 554,573 557,572 561,570 564,569 566,569 571,567 575,566 576,566 580,565 585,564 589,563 594,562 599,562 600,562 603,562 608,561 612,561 617,561 622,561 626,561 631,562 632,562 636,562 640,563 645,563 650,564 654,565 656,566 659,566 664,568 668,569 669,569 673,571 677,573 678,573 682,575 685,577 687,578 691,581 691,581 696,585 696,585 700,588 701,589 704,592 705,594 707,596 710,600 710,600 712,604 714,608 715,608 716,611 718,615 719,619 719,619 720,623 721,627 722,630 723,634 723,638 724,642 724,646 724,649 "/>
<polyline fill="none" opacity="1" stroke="#471F53" stroke-width="2" points="223,436 222,436 218,435 213,434 209,433 205,432 204,432 199,430 195,428 195,428 190,426 186,425 186,424 181,421 180,421 176,418 174,417 172,415 170,413 167,410 166,409 162,405 162,405 160,402 158,399 157,398 155,394 153,391 153,390 151,386 150,383 148,379 148,379 147,375 146,371 146,367 145,363 145,360 144,356 144,352 144,348 144,344 145,341 145,337 146,333 146,329 147,325 148,322 148,321 150,318 151,314 153,310 153,309 155,306 157,302 158,301 159,299 162,295 162,295 166,291 167,290 169,287 172,285 174,283 176,282 179,280 181,279 186,276 186,276 190,274 194,272 195,272 199,270 204,268 204,268 209,267 213,266 218,265 220,264 223,264 227,263 232,263 237,262 241,262 246,262 250,262 255,262 260,262 264,262 269,262 274,263 278,264 283,264 283,264 288,265 292,266 297,268 299,268 302,269 306,271 310,272 311,272 315,275 318,276 320,277 324,280 325,

### 2.3 Contour with Teal Isolines, Dark Background

In [14]:
evcxr_figure((640, 480), |root| {
    _ = root.fill(&RGBColor(20, 20, 30));

    let (xs, ys, z, cols, rows) = build_contour_grid(0.04);
    let levels = contour_levels();

    let mut chart = ChartBuilder::on(&root)
        .caption("Contour with Teal Isolines: Z = sin(X)·cos(Y)", ("Arial", 24).into_font().color(&WHITE))
        .margin(20)
        .x_label_area_size(50)
        .y_label_area_size(50)
        .build_cartesian_2d(-PI..PI, -PI..PI)?;

    chart.configure_mesh()
        .x_desc("X (radians)")
        .y_desc("Y (radians)")
        .label_style(("sans-serif", 16).into_font().color(&WHITE))
        .axis_desc_style(("sans-serif", 20).into_font().color(&WHITE))
        .draw()?;

    let teal = RGBColor(0, 220, 190);
    for &level in &levels {
        let segs = marching_squares_segments(&z, cols, rows, &xs, &ys, level);
        let chains = chain_segments(&segs);
        for chain in &chains {
            chart.draw_series(std::iter::once(PathElement::new(
                chain.clone(),
                teal.stroke_width(2),
            )))?;
        }
        if let Some(longest) = chains.iter().max_by_key(|c| c.len()) {
            let mid = longest[longest.len() / 2];
            chart.draw_series(std::iter::once(Text::new(
                format!("{:.1}", level),
                (mid.0, mid.1),
                ("sans-serif", 18).into_font().color(&WHITE),
            )))?;
        }
    }
    
    Ok(())
}).style("width:100%")

Contour with Teal Isolines: Z = sin(X)·cos(Y)
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
Y (radians)
 
 
X (radians)
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
-3.0
 
 
 
-2.0
 
 
 
-1.0
 
 
 
0.0
 
 
 
1.0
 
 
 
2.0
 
 
 
3.0
 
 
 
 
-3.0
 
 
 
-2.0
 
 
 
-1.0
 
 
 
0.0
 
 
 
1.0
 
 
 
2.0
 
 
 
3.0
 
 
<polyline fill="none" opacity="1" stroke="#00DCBE" stroke-width="2" points="425,409 425,407 425,405 426,403 426,402 427,400 427,398 429,396 429,394 430,394 432,391 433,390 434,389 436,387 436,386 439,384 440,384 442,382 443,381 446,380 447,380 450,378 452,378 454,377 457,376 460,375 461,375 464,374 468,374 471,373 475,373 475,373 478,373 482,373 485,373 487,373 489,373 492,373 496,374 499,374 503,375 503,375 506,376 510,377 511,378 513,379 516,380 517,380 520,382 520,382 524,384 524,384 527,387 527,387 529,389 531,391 531,391 533,394 534,396 534,397 535,398 536,400 537,403 537,405 537,407 537,409 "/>
<polyline fill="none" opacity="1" stroke="#00DCBE" stroke-width="2" points="192,266 191,266 188,265 185,264 181,263 181,263 178,262 175,261 174,261 171,259 170,259 167,257 166,256 164,255 163,254 160,252 160,252 158,250 157,248 156,247 155,245 154,243 153,242 153,240 152,238 151,236 151,234 151,231 151,229 151,227 151,224 152,222 152,220 153,218 153,218 155,215 156,213 157,212 158,211 160,208 160,208 163,206 164,205 166,204 167,203 170,202 171,201 174,199 174,199 178,198 181,197 181,197 185,196 188,195 190,195 192,194 195,194 199,194 202,193 206,193 209,193 213,193 216,194 220,194 223,195 223,195 227,195 230,196 233,197 234,197 237,198 239,199 241,200 244,202 244,202 248,204 248,204 251,206 251,206 253,208 255,210 256,211 257,213 258,214 259,215 260,218 261,220 262,222 262,222 262,224 263,227 263,229 263,231 263,234 262,236 262,238 262,238 261,240 260,243 259,245 258,246 257,247 255,250 255,250 253,252 251,254 251,254 248,256 247,256 244,258 244,259 241,260 239,261 237,262 234,263 232,263 230,264 227,265 223,265 222,266 220,266 216,266 213,267 209,267 206,267 202,267 199,266 195,266 192,266 "/>
 
 
 
-0.8
 
<polyline fill="none" opacity="1" stroke="#00DCBE" stroke-width="2" points="400,409 400,407 400,405 401,403 401,400 402,398 402,398 402,396 403,394 404,391 405,389 405,389 406,387 407,384 409,383 409,382 411,380 412,378 413,378 415,375 416,375 418,373 419,372 421,371 423,370 425,368 426,368 429,366 429,366 433,364 435,364 436,363 440,362 442,362 443,361 447,360 450,359 451,359 454,359 457,358 461,358 464,357 468,357 469,357 471,357 475,357 478,357 482,357 485,357 489,357 492,357 493,357 496,357 499,357 503,358 506,358 510,359 511,359 513,360 517,360 520,361 521,362 524,362 527,364 528,364 531,365 533,366 534,367 538,368 538,368 541,371 541,371 545,373 545,373 547,375 548,376 550,378 552,380 552,380 554,382 555,384 555,385 556,387 558,389 559,391 559,391 560,394 560,396 561,398 561,400 562,403 562,405 562,407 562,409 "/>
<polyline fill="none" opacity="1" stroke="#00DCBE" stroke-width="2" points="185,282 184,282 181,281 178,280 174,280 172,279 171,279 167,278 164,277 164,277 160,276 158,275 157,274 153,273 153,272 150,271 148,270 146,269 145,268 143,266 142,266 139,263 139,263 137,261 136,259 135,259 134,256 132,254 132,254 131,252 130,250 129,247 129,247 128,245 127,243 127,240 126,238 126,236 126,234 126,231 126,229 126,227 126,224 126,222 127,220 127,218 128,215 129,213 129,213 130,211 131,208 132,206 132,206 134,204 135,202 136,201 137,199 139,197 139,197 142,195 143,194 145,192 146,191 148,190 150,189 152,188 153,187 157,186 157,186 160,184 163,183 164,183 167,182 171,181 171,181 174,180 178,180 181,179 183,179 185,178 188,178 192,178 195,177 199,177 202,177 206,177 209,177 213,177 216,177 220,178 223,178 227,178 230,179 230,179 234,179 237,180 241,181 242,181 244,181 248,182 250,183 251,184 255,185 256,186 258,186 261,188 2

### 2.4 Filled Contour: Inferno Colourmap

In [15]:
evcxr_figure((640, 480), |root| {
    _ = root.fill(&RGBColor(20, 20, 30));

    let (xs, ys, z, cols, rows) = build_contour_grid(0.04);
    let levels = contour_levels();

    let mut chart = ChartBuilder::on(&root)
        .caption("Filled Contour: Z = sin(X)·cos(Y)", ("Arial", 24).into_font().color(&WHITE))
        .margin(20)
        .x_label_area_size(50)
        .y_label_area_size(50)
        .build_cartesian_2d(-PI..PI, -PI..PI)?;

    chart.configure_mesh()
        .x_desc("X (radians)")
        .y_desc("Y (radians)")
        .label_style(("sans-serif", 16).into_font().color(&WHITE))
        .axis_desc_style(("sans-serif", 20).into_font().color(&WHITE))
        .draw()?;

    let rects: Vec<Rectangle<(f64, f64)>> = (0..rows - 1)
        .flat_map(|j| {
            let (z, xs, ys) = (&z, &xs, &ys);
            (0..cols - 1).map(move |i| {
                let val = (z[j * cols + i]
                    + z[j * cols + i + 1]
                    + z[(j + 1) * cols + i]
                    + z[(j + 1) * cols + i + 1])
                    / 4.0;
                // Z ∈ [-1, 1] → t ∈ [0, 1]
                let t = ((val + 1.0) / 2.0).clamp(0.0, 1.0);
                Rectangle::new(
                    [(xs[i], ys[j]), (xs[i + 1], ys[j + 1])],
                    inferno_cmap(t).filled(),
                )
            })
        })
        .collect();
    chart.draw_series(rects)?;
    
    Ok(())
}).style("width:100%")

Filled Contour: Z = sin(X)·cos(Y)
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
Y (radians)
 
 
X (radians)
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
-3.0
 
 
 
-2.0
 
 
 
-1.0
 
 
 
0.0
 
 
 
1.0
 
 
 
2.0
 
 
 
3.0
 
 
 
 
-3.0
 
 
 
-2.0
 
 
 
-1.0
 
 
 
0.0
 
 
 
1.0
 
 
 
2.0
 
 
 
3.0
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 


## 3. Stream Plots

### Helpers 

In [16]:
fn bilerp(
    grid: &[f64],
    cols: usize,
    rows: usize,
    x: f64,
    y: f64,
    x_min: f64,
    x_max: f64,
    y_min: f64,
    y_max: f64,
) -> f64 {
    let fx = (x - x_min) / (x_max - x_min) * (cols as f64 - 1.0);
    let fy = (y - y_min) / (y_max - y_min) * (rows as f64 - 1.0);
    let ix = (fx.floor() as usize).min(cols - 2);
    let iy = (fy.floor() as usize).min(rows - 2);
    let tx = fx - ix as f64;
    let ty = fy - iy as f64;
    let v00 = grid[iy * cols + ix];
    let v10 = grid[iy * cols + ix + 1];
    let v01 = grid[(iy + 1) * cols + ix];
    let v11 = grid[(iy + 1) * cols + ix + 1];
    lerp(lerp(v00, v10, tx), lerp(v01, v11, tx), ty)
}

/// Velocity field: saddle point flow.
struct VectorField {
    x_min: f64,
    x_max: f64,
    y_min: f64,
    y_max: f64,
    cols: usize,
    rows: usize,
    u: Vec<f64>,
    v: Vec<f64>,
}

impl VectorField {
    fn new() -> Self {
        let (cols, rows) = (350, 250);
        let (x_min, x_max, y_min, y_max) = (-4.0, 4.0, -4.0, 4.0);
        let mut u = vec![0.0f64; cols * rows];
        let mut v = vec![0.0f64; cols * rows];
        for j in 0..rows {
            let y = y_min + (y_max - y_min) * j as f64 / (rows - 1) as f64;
            for i in 0..cols {
                let x = x_min + (x_max - x_min) * i as f64 / (cols - 1) as f64;
                // Saddle-point / hyperbolic flow
                u[j * cols + i] = x - y;
                v[j * cols + i] = -x + y;
            }
        }
        VectorField {
            x_min,
            x_max,
            y_min,
            y_max,
            cols,
            rows,
            u,
            v,
        }
    }

    fn sample_uv(&self, x: f64, y: f64) -> (f64, f64) {
        (
            bilerp(
                &self.u, self.cols, self.rows, x, y, self.x_min, self.x_max, self.y_min, self.y_max,
            ),
            bilerp(
                &self.v, self.cols, self.rows, x, y, self.x_min, self.x_max, self.y_min, self.y_max,
            ),
        )
    }

    /// RK4 streamline integration (normalised step for uniform line density).
    fn streamline(&self, x0: f64, y0: f64, steps: usize, dt: f64) -> Vec<(f64, f64)> {
        let mut pts = vec![(x0, y0)];
        let (mut x, mut y) = (x0, y0);
        for _ in 0..steps {
            let (u1, v1) = self.sample_uv(x, y);
            let m1 = (u1 * u1 + v1 * v1).sqrt().max(1e-12);
            let (dx1, dy1) = (u1 / m1 * dt, v1 / m1 * dt);

            let (u2, v2) = self.sample_uv(x + dx1 * 0.5, y + dy1 * 0.5);
            let m2 = (u2 * u2 + v2 * v2).sqrt().max(1e-12);
            let (dx2, dy2) = (u2 / m2 * dt, v2 / m2 * dt);

            let (u3, v3) = self.sample_uv(x + dx2 * 0.5, y + dy2 * 0.5);
            let m3 = (u3 * u3 + v3 * v3).sqrt().max(1e-12);
            let (dx3, dy3) = (u3 / m3 * dt, v3 / m3 * dt);

            let (u4, v4) = self.sample_uv(x + dx3, y + dy3);
            let m4 = (u4 * u4 + v4 * v4).sqrt().max(1e-12);
            let (dx4, dy4) = (u4 / m4 * dt, v4 / m4 * dt);

            x += (dx1 + 2.0 * dx2 + 2.0 * dx3 + dx4) / 6.0;
            y += (dy1 + 2.0 * dy2 + 2.0 * dy3 + dy4) / 6.0;

            if x < self.x_min || x > self.x_max || y < self.y_min || y > self.y_max {
                break;
            }
            pts.push((x, y));
        }
        pts
    }
}

fn seed_grid(nx: usize, ny: usize, f: &VectorField) -> Vec<(f64, f64)> {
    (0..ny)
        .flat_map(|j| {
            let y = f.y_min + (f.y_max - f.y_min) * (j as f64 + 0.5) / ny as f64;
            (0..nx).map(move |i| {
                let x = f.x_min + (f.x_max - f.x_min) * (i as f64 + 0.5) / nx as f64;
                (x, y)
            })
        })
        .collect()
}

fn streamline_arrow_head(
    p_start: (f64, f64),
    p_end: (f64, f64),
    color: impl Color,
    scale: f64,
) -> Polygon<(f64, f64)> {
    let dx = p_end.0 - p_start.0;
    let dy = p_end.1 - p_start.1;
    let mag = (dx * dx + dy * dy).sqrt().max(1e-9);
    let nx = dx / mag;
    let ny = dy / mag;
    let (px, py) = (-ny, nx);
    let s = scale;
    let hx = p_end.0 - nx * s;
    let hy = p_end.1 - ny * s;
    let hw = s * 0.45;
    Polygon::new(
        vec![
            p_end,
            (hx + px * hw, hy + py * hw),
            (hx - px * hw, hy - py * hw),
        ],
        color.filled(),
    )
}

### 3.1 Simple Stream Plot

In [17]:
evcxr_figure((640, 480), |root| {
    _ = root.fill(&WHITE);

    let field = VectorField::new();
    let seeds = seed_grid(16, 12, &field);

    let mut chart = ChartBuilder::on(&root)
        .caption("Stream Plot: Saddle-Point Flow (U = X-Y, V = -X+Y)", ("Arial", 24).into_font().color(&BLACK))
        .margin(20)
        .x_label_area_size(50)
        .y_label_area_size(50)
        .build_cartesian_2d(-4.0f64..4.0, -4.0f64..4.0)?;

    chart.configure_mesh()
        .x_desc("X")
        .y_desc("Y")
        .label_style(("sans-serif", 16).into_font().color(&BLACK))
        .axis_desc_style(("sans-serif", 20).into_font().color(&BLACK))
        .draw()?;

    let stream_color = RGBColor(30, 90, 200);
    for (sx, sy) in &seeds {
        let line = field.streamline(*sx, *sy, 400, 0.06);
        if line.len() > 1 {
            chart.draw_series(std::iter::once(PathElement::new(
                line.clone(),
                stream_color.stroke_width(1),
            )))?;
            let mid = line.len() / 2;
            chart.draw_series(std::iter::once(streamline_arrow_head(
                line[mid - 1],
                line[mid],
                stream_color,
                0.2,
            )))?;
        }
    }
    
    Ok(())
}).style("width:100%")

Stream Plot: Saddle-Point Flow (U = X-Y, V = -X+Y)
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
Y
 
 
X
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
-4.0
 
 
 
-3.0
 
 
 
-2.0
 
 
 
-1.0
 
 
 
0.0
 
 
 
1.0
 
 
 
2.0
 
 
 
3.0
 
 
 
4.0
 
 
 
 
-4.0
 
 
 
-3.0
 
 
 
-2.0
 
 
 
-1.0
 
 
 
0.0
 
 
 
1.0
 
 
 
2.0
 
 
 
3.0
 
 
 
4.0
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
<polyline fill="none" opacity="1" stroke="#1E5AC8" stroke-width="1" points="258,305 261,307 264,309 267,311 270,312 273,314 276,316 279,318 282,320 284,322 287,324 290,326 293,328 296,330 299,331 302,333 305,335 308,337 311,339 314,341 316,343 319,345 322,347 325,349 328,350 331,352 334,354 337,356 340,358 343,360 346,362 348,364 351,366 354,368 357,370 360,371 363,373 366,375 369,377 372,379 375,381 378,383 381,385 383,387 386,389 389,390 392,392 395,394 398,396 401,398 404,400 407,402 410,404 413,406 415,408 "/>
 
<polyline fill="none" opacity="1" stroke="#1E5AC8" stroke-width="1" points="293,305 295,307 298,309 301,311 304,312 307,314 310,316 313,318 316,320 319,322 322,324 325,326 327,328 330,330 333,331 336,333 339,335 342,337 345,339 348,341 351,343 354,345 357,347 359,349 362,350 365,352 368,354 371,356 374,358 377,360 380,362 383,364 386,366 389,368 392,370 394,371 397,373 400,375 403,377 406,379 409,381 412,383 415,385 418,387 421,389 424,390 426,392 429,394 432,396 435,398 438,400 441,402 444,404 447,406 450,408 "/>
 
<polyline fill="none" opacity="1" stroke="#1E5AC8" stroke-width="1" points="327,305 330,307 333,309 336,311 338,312 341,314 344,316 347,318 350,320 353,322 356,324 359,326 362,328 365,330 368,331 371,333 373,335 376,337 379,339 382,341 385,343 388,345 391,347 394,349 397,350 400,352 403,354 405,356 408,358 411,360 414,362 417,364 420,366 423,368 426,370 429,371 432,373 435,375 437,377 440,379 443,381 446,383 449,385 452,387 455,389 458,390 461,392 464,394 467,396 470,398 472,400 475,402 478,404 481,406 484,408 "/>
 
<polyline fill="none" opacity="1" stroke="#1E5AC8" stroke-width="1" points="361,305 364,307 367,309 370,311 373,312 376,314 379,316 382,318 384,320 387,322 390,324 393,326 396,328 399,330 402,331 405,333 408,335 411,337 414,339 416,341 419,343 422,345 425,347 428,349 431,350 434,352 437,354 440,356 443,358 446,360 449,362 451,364 454,366 457,368 460,370 463,371 466,373 469,375 472,377 475,379 478,381 481,383 483,385 486,387 489,389 492,390 495,392 498,394 501,396 504,398 507,400 510,402 513,404 515,406 518,408 "/>
 
<polyline fill="none" opacity="1" stroke="#1E5AC8" stroke-width="1" points="395,305 398,307 401,309 404,311 407,312 410,314 413,316 416,318 419,320 422,322 425,324 427,326 430,328 433,330 436,331 439,333 442,335 445,337 448,339 451,341 454,343 457,345 460,347 462,349 465,350 468,352 471,354 474,356 477,358 480,360 483,362 486,364 489,366 492,368 494,370 497,371 500,373 503,375 506,377 509,379 512,381 515,383 518,385 521,387 524,389 526,390 529,392 532,394 535,396 538,398 541,400 544,402 547,404 550,406 553,408 "/>
 
<polyline fill="none" opacity="1" stroke="#1E5AC8" stroke-width="1" points="430,305 433,307 436,309 439,311 441,312 444,314 447,316 450,318 453,320 456,322 459,324 462,326 465,328 468,330 471,331 473,333 476,335 479,337 482,339 485,341 488,343 491,345 494,347 497,349 500,350 503,352 505,354 508,356 511,358 514,360 517,362 520,364 523,366 526,368 529,370 532,371 535,373 538,375 540,377 543,379 546,381 549,383 552,385 555,387 558,389 561,390 564,392 567,394 570,396 572,398 575,400 578,402 581,404 584,406 587,408 "/>
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 

### 3.2 Sparse Seed Stream Plot

In [18]:
evcxr_figure((640, 480), |root| {
    _ = root.fill(&WHITE);

    let field = VectorField::new();
    let seeds = seed_grid(8, 6, &field);
    
    let mut chart = ChartBuilder::on(&root)
        .caption("Stream Plot: Variable Density (sparse seeds)", ("Arial", 24).into_font().color(&BLACK))
        .margin(20)
        .x_label_area_size(50)
        .y_label_area_size(50)
        .build_cartesian_2d(-4.0f64..4.0, -4.0f64..4.0)?;

    chart.configure_mesh()
        .x_desc("X")
        .y_desc("Y")
        .label_style(("sans-serif", 16).into_font().color(&BLACK))
        .axis_desc_style(("sans-serif", 20).into_font().color(&BLACK))
        .draw()?;

    let stream_color = RGBColor(180, 30, 120);
    for (sx, sy) in &seeds {
        let line = field.streamline(*sx, *sy, 400, 0.06);
        if line.len() > 1 {
            chart.draw_series(std::iter::once(PathElement::new(
                line.clone(),
                stream_color.stroke_width(2),
            )))?;
            let mid = line.len() / 2;
            chart.draw_series(std::iter::once(streamline_arrow_head(
                line[mid - 1],
                line[mid],
                stream_color,
                0.25,
            )))?;
        }
    }
    
    Ok(())
}).style("width:100%")

Stream Plot: Variable Density (sparse seeds)
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
Y
 
 
X
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
-4.0
 
 
 
-3.0
 
 
 
-2.0
 
 
 
-1.0
 
 
 
0.0
 
 
 
1.0
 
 
 
2.0
 
 
 
3.0
 
 
 
4.0
 
 
 
 
-4.0
 
 
 
-3.0
 
 
 
-2.0
 
 
 
-1.0
 
 
 
0.0
 
 
 
1.0
 
 
 
2.0
 
 
 
3.0
 
 
 
4.0
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
<polyline fill="none" opacity="1" stroke="#B41E78" stroke-width="2" points="241,260 238,258 235,256 232,254 229,252 227,250 224,248 221,247 218,245 215,243 212,241 209,239 206,237 203,235 200,233 197,231 194,229 192,228 189,226 186,224 183,222 180,220 177,218 174,216 171,214 168,212 165,210 162,209 160,207 157,205 154,203 151,201 148,199 145,197 142,195 139,193 136,191 133,189 130,188 128,186 125,184 122,182 119,180 116,178 113,176 110,174 107,172 104,170 101,169 98,167 95,165 93,163 90,161 87,159 84,157 81,155 78,153 75,151 72,149 "/>
 
<polyline fill="none" opacity="1" stroke="#B41E78" stroke-width="2" points="310,260 313,262 316,264 318,266 321,268 324,269 327,271 330,273 333,275 336,277 339,279 342,281 345,283 348,285 350,287 353,288 356,290 359,292 362,294 365,296 368,298 371,300 374,302 377,304 380,306 382,308 385,309 388,311 391,313 394,315 397,317 400,319 403,321 406,323 409,325 412,327 415,328 417,330 420,332 423,334 426,336 429,338 432,340 435,342 438,344 441,346 444,347 447,349 449,351 452,353 455,355 458,357 461,359 464,361 467,363 470,365 473,367 476,368 479,370 481,372 484,374 487,376 490,378 493,380 496,382 499,384 502,386 505,387 508,389 511,391 513,393 516,395 519,397 522,399 525,401 528,403 531,405 534,407 537,408 "/>
 
<polyline fill="none" opacity="1" stroke="#B41E78" stroke-width="2" points="378,260 381,262 384,264 387,266 390,268 393,269 396,271 399,273 402,275 405,277 407,279 410,281 413,283 416,285 419,287 422,288 425,290 428,292 431,294 434,296 437,298 439,300 442,302 445,304 448,306 451,308 454,309 457,311 460,313 463,315 466,317 469,319 471,321 474,323 477,325 480,327 483,328 486,330 489,332 492,334 495,336 498,338 501,340 504,342 506,344 509,346 512,347 515,349 518,351 521,353 524,355 527,357 530,359 533,361 536,363 538,365 541,367 544,368 547,370 550,372 553,374 556,376 559,378 562,380 565,382 568,384 570,386 573,387 576,389 579,391 582,393 585,395 588,397 591,399 594,401 597,403 600,405 602,407 605,408 "/>
 
<polyline fill="none" opacity="1" stroke="#B41E78" stroke-width="2" points="447,260 450,262 453,264 456,266 459,268 461,269 464,271 467,273 470,275 473,277 476,279 479,281 482,283 485,285 488,287 491,288 494,290 496,292 499,294 502,296 505,298 508,300 511,302 514,304 517,306 520,308 523,309 526,311 528,313 531,315 534,317 537,319 540,321 543,323 546,325 549,327 552,328 555,330 558,332 560,334 563,336 566,338 569,340 572,342 575,344 578,346 581,347 584,349 587,351 590,353 593,355 595,357 598,359 601,361 604,363 607,365 610,367 613,368 616,370 "/>
 
 
 
 
 
 
 
 
 
<polyline fill="none" opacity="1" stroke="#B41E78" stroke-width="2" points="241,200 238,198 235,196 232,194 229,192 227,191 224,189 221,187 218,185 215,183 212,181 209,179 206,177 203,175 200,173 197,172 194,170 192,168 189,166 186,164 183,162 180,160 177,158 174,156 171,154 168,152 165,151 162,149 160,147 157,145 154,143 151,141 148,139 145,137 142,135 139,133 136,132 133,130 130,128 128,126 125,124 122,122 119,120 116,118 113,116 110,114 107,113 104,111 101,109 98,107 95,105 93,103 90,101 87,99 84,97 81,95 78,93 75,92 72,90 "/>
 
<polyline fill="none" opacity="1" stroke="#B41E78" stroke-width="2" points="310,200 307,198 304,196 301,194 298,192 295,191 292,189 289,187 286,185 283,183 281,181 278,179 275,177 272,175 269,173 266,172 263,170 260,168 257,166 254,164 2

### 3.3 Colour By Speed Stream Plot

In [19]:
evcxr_figure((640, 480), |root| {
    _ = root.fill(&RGBColor(10, 10, 20));

    let field = VectorField::new();
    let seeds = seed_grid(18, 14, &field);
    
    let mut chart = ChartBuilder::on(&root)
        .caption("Stream Plot: Colour by V-Magnitude", ("Arial", 24).into_font().color(&WHITE))
        .margin(20)
        .x_label_area_size(50)
        .y_label_area_size(50)
        .build_cartesian_2d(-4.0f64..4.0, -4.0f64..4.0)?;

    chart.configure_mesh()
        .x_desc("X")
        .y_desc("Y")
        .axis_desc_style(("sans-serif", 20).into_font().color(&WHITE))
        .label_style(("sans-serif", 16).into_font().color(&WHITE))
        .draw()?;

    // |V| = |-X+Y| ∈ [0, ~5.66] over the grid
    let v_max = 5.7f64;

    for (sx, sy) in &seeds {
        let line = field.streamline(*sx, *sy, 400, 0.06);
        if line.len() < 2 {
            continue;
        }
        for pair in line.windows(2) {
            let mx = (pair[0].0 + pair[1].0) / 2.0;
            let my = (pair[0].1 + pair[1].1) / 2.0;
            let v_val = (-mx + my).abs();
            let t = (v_val / v_max).clamp(0.0, 1.0);
            chart.draw_series(std::iter::once(PathElement::new(
                vec![pair[0], pair[1]],
                viridis_cmap(t).stroke_width(1),
            )))?;
        }
        let mid = line.len() / 2;
        let p0 = line[mid - 1];
        let p1 = line[mid];
        let mx = (p0.0 + p1.0) / 2.0;
        let my = (p0.1 + p1.1) / 2.0;
        let v_val = (-mx + my).abs();
        let t = (v_val / v_max).clamp(0.0, 1.0);
        chart.draw_series(std::iter::once(streamline_arrow_head(
            p0,
            p1,
            viridis_cmap(t),
            0.2,
        )))?;
    }

    Ok(())
}).style("width:100%")

Stream Plot: Colour by V-Magnitude
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
Y
 
 
X
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
-4.0
 
 
 
-3.0
 
 
 
-2.0
 
 
 
-1.0
 
 
 
0.0
 
 
 
1.0
 
 
 
2.0
 
 
 
3.0
 
 
 
4.0
 
 
 
 
-4.0
 
 
 
-3.0
 
 
 
-2.0
 
 
 
-1.0
 
 
 
0.0
 
 
 
1.0
 
 
 
2.0
 
 
 
3.0
 
 
 
4.0
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 

### 3.4 Variable Width Stream Plot

In [20]:
evcxr_figure((640, 480), |root| {
    _ = root.fill(&WHITE);

    let field = VectorField::new();
    let seeds = seed_grid(12, 9, &field);
    
    let mut chart = ChartBuilder::on(&root)
        .caption("Stream Plot: Variable Line Width (by speed)", ("Arial", 24).into_font().color(&BLACK))
        .margin(20)
        .x_label_area_size(50)
        .y_label_area_size(50)
        .build_cartesian_2d(-4.0f64..4.0, -4.0f64..4.0)?;

    chart.configure_mesh()
        .x_desc("X")
        .y_desc("Y")
        .label_style(("sans-serif", 16).into_font().color(&BLACK))
        .axis_desc_style(("sans-serif", 20).into_font().color(&BLACK))
        .draw()?;

    let speed_max = 5.7f64;
    for (sx, sy) in &seeds {
        let line = field.streamline(*sx, *sy, 400, 0.06);
        if line.len() < 2 {
            continue;
        }
        for pair in line.windows(2) {
            let mx = (pair[0].0 + pair[1].0) / 2.0;
            let my = (pair[0].1 + pair[1].1) / 2.0;
            let (u, v) = field.sample_uv(mx, my);
            let speed = (u * u + v * v).sqrt();
            let lw = ((speed / speed_max) * 4.0 + 0.5).clamp(1.0, 4.0) as u32;
            chart.draw_series(std::iter::once(PathElement::new(
                vec![pair[0], pair[1]],
                RGBColor(40, 40, 60).stroke_width(lw),
            )))?;
        }
        let mid = line.len() / 2;
        chart.draw_series(std::iter::once(streamline_arrow_head(
            line[mid - 1],
            line[mid],
            RGBColor(40, 40, 60),
            0.2,
        )))?;
    }
    
    Ok(())
}).style("width:100%")

Stream Plot: Variable Line Width (by speed)
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
Y
 
 
X
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
-4.0
 
 
 
-3.0
 
 
 
-2.0
 
 
 
-1.0
 
 
 
0.0
 
 
 
1.0
 
 
 
2.0
 
 
 
3.0
 
 
 
4.0
 
 
 
 
-4.0
 
 
 
-3.0
 
 
 
-2.0
 
 
 
-1.0
 
 
 
0.0
 
 
 
1.0
 
 
 
2.0
 
 
 
3.0
 
 
 
4.0
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 


## 4. Quiver Plot

In [21]:
evcxr_figure((640, 480), |root| {
    _ = root.fill(&WHITE);

    let grid_step = 0.4f64;
    let xs: Vec<f64> = {
        let mut v = Vec::new();
        let mut x = -4.0;
        while x <= 4.0 {
            v.push(x);
            x += grid_step;
        }
        v
    };
    let ys = xs.clone();
    
    let mut chart = ChartBuilder::on(&root)
        .caption("Quiver Plot: U=cos(X+Y), V=sin(X-Y)", ("Arial", 24).into_font().color(&BLACK))
        .margin(20)
        .x_label_area_size(50)
        .y_label_area_size(50)
        .build_cartesian_2d(-4.0f64..4.0, -4.0f64..4.0)?;

    chart.configure_mesh()
        .x_desc("X")
        .y_desc("Y")
        .label_style(("sans-serif", 16).into_font().color(&BLACK))
        .axis_desc_style(("sans-serif", 20).into_font().color(&BLACK))
        .draw()?;

    let scale = 0.20f64;
    let head_len = 0.24f64;

    for &yv in &ys {
        for &xv in &xs {
            let u = (xv + yv).cos();
            let v = (xv - yv).sin();
            let speed = (u * u + v * v).sqrt().max(1e-9);
            let t = (speed / 1.414f64).clamp(0.0, 1.0);
            let color = plasma_cmap(t);

            let x_end = xv + u * scale;
            let y_end = yv + v * scale;

            chart.draw_series(std::iter::once(PathElement::new(
                vec![(xv, yv), (x_end, y_end)],
                color.stroke_width(1),
            )))?;

            if speed > 0.05 {
                let dx = u / speed;
                let dy = v / speed;
                let (px, py) = (-dy, dx);
                let s = scale * head_len;
                let hx = x_end - dx * s;
                let hy = y_end - dy * s;
                let hw = s * 0.45;
                chart.draw_series(std::iter::once(Polygon::new(
                    vec![
                        (x_end, y_end),
                        (hx + px * hw, hy + py * hw),
                        (hx - px * hw, hy - py * hw),
                    ],
                    color.filled(),
                )))?;
            }
        }
    }

    Ok(())
}).style("width:100%")

Quiver Plot: U=cos(X+Y), V=sin(X-Y)
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
Y
 
 
X
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
-4.0
 
 
 
-3.0
 
 
 
-2.0
 
 
 
-1.0
 
 
 
0.0
 
 
 
1.0
 
 
 
2.0
 
 
 
3.0
 
 
 
4.0
 
 
 
 
-4.0
 
 
 
-3.0
 
 
 
-2.0
 
 
 
-1.0
 
 
 
0.0
 
 
 
1.0
 
 
 
2.0
 
 
 
3.0
 
 
 
4.0